# Production Tuning — The Experiments That Separate Expert from Competent
Most tutorials pick a chunk size and move on. Experts *measure* the effect of chunk size, overlap, and k on real retrieval quality. This notebook runs those experiments and does failure analysis. Fully offline.

In [ ]:
# Self-contained mock embedder + LLM so this notebook runs with NO api key / NO network.
# Swap MockEmbedder -> your InHouseEmbeddings and mock_llm -> your ask() for real use.
import numpy as np, re
from collections import Counter

class MockEmbedder:
    """Deterministic bag-of-words embedding: same words -> similar vectors.
    Good enough to demonstrate retrieval behavior without a real model."""
    def __init__(self, dim=64):
        self.dim = dim
    def _vec(self, text):
        rng = np.random.default_rng(0)
        base = {}
        v = np.zeros(self.dim)
        for w in re.findall(r"\w+", text.lower()):
            h = abs(hash(w)) % self.dim
            v[h] += 1.0
        n = np.linalg.norm(v)
        return v / n if n > 0 else v
    def embed_documents(self, texts): return [self._vec(t).tolist() for t in texts]
    def embed_query(self, text): return self._vec(text).tolist()

embedder = MockEmbedder()

def mock_llm(system, user, **kw):
    """Extremely dumb stand-in: echoes retrieved context if present, else says IDK.
    Replace with your real ask(system, user, model=...) helper for genuine answers."""
    if "no relevant" in user.lower() or "context: \n\nquestion" in user.lower():
        return "I don't know based on the provided context."
    # pull the 'Context:' block back out as a fake 'answer'
    m = re.search(r"Context:(.*?)Question:", user, re.S)
    ctx = m.group(1).strip() if m else ""
    return f"(mock answer grounded in retrieved context) {ctx[:160]}"

## 1. Chunk size experiment — the single highest-impact knob

In [ ]:
import numpy as np

# A longer document to chunk different ways
DOC = ("""
Refund policy. Customers may request a refund within 30 days of purchase.
Refunds are processed within 5 business days to the original payment method.
A restocking fee of 10 percent applies to opened electronics.
Shipping costs are non-refundable except when the item was defective.
For defective items, we cover return shipping and issue a full refund.
Warranty claims are separate from refunds and require the original receipt.
""").strip()

def chunk_fixed(text, size, overlap):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start+size]))
        start += size - overlap
    return [c for c in chunks if c]

QUERY = "how long do refunds take to process"
GOLD_PHRASE = "5 business days"   # the chunk we WANT retrieved must contain this

for size in [8, 15, 30, 60]:
    chunks = chunk_fixed(DOC, size, overlap=2)
    mat = np.array(embedder.embed_documents(chunks))
    q = np.array(embedder.embed_query(QUERY))
    best_idx = int(np.argmax(mat @ q))
    hit = GOLD_PHRASE in chunks[best_idx]
    print(f"chunk_size={size:3d} | n_chunks={len(chunks):2d} | top chunk contains answer: {hit}")
    print(f"           top chunk: {chunks[best_idx][:70]}...")

### Observe
- Very small chunks: the answer phrase may be isolated but lack context.
- Very large chunks: the answer is buried with unrelated text, diluting the embedding.
- There's a sweet spot. In production you'd sweep this against a labeled benchmark (previous notebook) rather than eyeballing one query.

## 2. Overlap experiment — does it rescue boundary answers?

In [ ]:
# An answer that sits RIGHT at a chunk boundary
DOC2 = "word " * 20 + "refunds take exactly five business days to process " + "word " * 20

for overlap in [0, 3, 8]:
    chunks = chunk_fixed(DOC2.strip(), size=15, overlap=overlap)
    mat = np.array(embedder.embed_documents(chunks))
    q = np.array(embedder.embed_query("how long do refunds take"))
    best = chunks[int(np.argmax(mat @ q))]
    intact = "five business days" in best
    print(f"overlap={overlap}: answer phrase intact in top chunk: {intact}")

### Observe
Overlap exists specifically to stop answers being severed at chunk boundaries. Too little overlap = boundary answers get split and retrieval misses them. Too much = redundant storage and near-duplicate chunks. 10-20% of chunk size is the usual starting point.

## 3. Retrieval failure analysis — the expert debugging habit

In [ ]:
CORPUS = {
    "d1": "Refunds are processed within 5 business days.",
    "d2": "Shipping takes 3 to 5 business days.",
    "d3": "Warranty covers defects for 12 months.",
}
ids = list(CORPUS); texts = list(CORPUS.values())
mat = np.array(embedder.embed_documents(texts))

def diagnose(query, expected_id):
    q = np.array(embedder.embed_query(query))
    sims = mat @ q
    order = np.argsort(sims)[::-1]
    print(f"Query: {query!r}  (expected top hit: {expected_id})")
    for rank, i in enumerate(order, 1):
        marker = " <-- EXPECTED" if ids[i] == expected_id else ""
        print(f"  rank {rank}: {ids[i]} sim={sims[i]:.3f}{marker}")
    got = ids[order[0]]
    if got != expected_id:
        print("  FAILURE: wrong doc ranked first.")
        print("  Likely causes: query/doc vocabulary mismatch, chunk too broad,")
        print("  or the two docs are genuinely near-synonymous (see d1 vs d2 — both mention 'business days').")
    print()

diagnose("how long do refunds take", "d1")
diagnose("how many business days for delivery", "d2")  # note d1 also says 'business days'

### Observe
This is the single most useful production habit: when an answer is wrong, look at the *ranked retrieval list with scores*, not just the final answer. Most 'bad LLM answer' bugs are actually retrieval bugs — the LLM answered correctly from the wrong chunk. `d1` and `d2` both containing 'business days' is exactly the kind of near-collision that causes silent failures.

## Your turn — the capstone tuning exercise
Combine this notebook with the evaluation harness:
1. Take a real labeled benchmark (15+ queries with known relevant docs).
2. Sweep chunk_size ∈ {128, 256, 512} × overlap ∈ {0, 10%, 20%} × k ∈ {1,3,5}.
3. Compute mean NDCG@k for every combination.
4. Pick the config with the best NDCG, not the one that 'feels right'.
**This data-driven tuning loop is what separates an expert from someone who copied a chunk_size=500 from a tutorial.**